In [ ]:
# finalizer.py
import os
import time
import warnings
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, average_precision_score, brier_score_loss

warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

from core_glaucogate import FundusDataset, ConvNeXtBaseModel, ConvNeXtProGlaucoGate
from step3_train_eval import compute_ece, compute_selective_metrics, compute_all_metrics

class FastExternalFundusDataset(Dataset):
    """Fast CPU read + tight FOV crop only (zero slow CPU Gaussian blurs)."""
    def __init__(self, csv_path, global_size=512, highres_size=768):
        self.df = pd.read_csv(csv_path)
        self.global_size = global_size
        self.highres_size = highres_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_bgr = cv2.imread(str(row["image_path"]))
        if img_bgr is None:
            img_rgb = np.zeros((self.highres_size, self.highres_size, 3), dtype=np.uint8)
        else:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        gray = img_rgb[:, :, 1]
        mask = (gray > 14).astype(np.uint8)
        coords = cv2.findNonZero(mask)
        if coords is not None and len(coords) > 50:
            x, y, w, h = cv2.boundingRect(coords)
            if w > 0.3 * img_rgb.shape[1] and h > 0.3 * img_rgb.shape[0]:
                img_rgb = img_rgb[y:y+h, x:x+w]

        img_highres = cv2.resize(img_rgb, (self.highres_size, self.highres_size), interpolation=cv2.INTER_AREA)
        img_global = cv2.resize(img_highres, (self.global_size, self.global_size), interpolation=cv2.INTER_AREA)

        highres_t = torch.from_numpy(img_highres).permute(2, 0, 1).float() / 255.0
        global_t = torch.from_numpy(img_global).permute(2, 0, 1).float() / 255.0

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return {
            "image_id": str(row["image_id"]),
            "patient_id": str(row.get("patient_id", row["image_id"])),
            "global_img": (global_t - mean) / std,
            "highres_img": (highres_t - mean) / std,
            "raw_global": global_t,
            "label": torch.tensor(float(row["label"]), dtype=torch.float32),
            "od_gt": torch.tensor([
                float(row.get("od_cx", np.nan)),
                float(row.get("od_cy", np.nan)),
                float(row.get("od_r", np.nan))
            ], dtype=torch.float32)
        }

@torch.no_grad()
def gpu_detect_onh_and_normalize(x_glob, x_highres, raw_glob, od_gt):
    B = x_glob.shape[0]
    device = x_glob.device

    rg = 0.55 * raw_glob[:, 0:1] + 0.45 * raw_glob[:, 1:2]
    rg_32 = F.adaptive_avg_pool2d(rg, (32, 32)).squeeze(1)
    yy, xx = torch.meshgrid(
        torch.linspace(-1.0, 1.0, 32, device=device),
        torch.linspace(-1.0, 1.0, 32, device=device),
        indexing="ij"
    )
    valid_ring = (((xx**2 + yy**2) < 0.50) & (yy.abs() < 0.42)).float().unsqueeze(0)
    score = (rg_32 * valid_ring).view(B, -1)
    max_idx = torch.argmax(score, dim=1)
    py = (max_idx // 32).float() / 31.0 * 2.0 - 1.0
    px = (max_idx % 32).float() / 31.0 * 2.0 - 1.0
    auto_od = torch.stack([torch.clamp(px, -0.55, 0.55), torch.clamp(py, -0.40, 0.40), torch.full_like(px, 0.16)], dim=1)

    valid_gt = ~torch.isnan(od_gt[:, 0])
    final_od = torch.where(valid_gt.unsqueeze(1), od_gt, auto_od)

    for x in (x_glob, x_highres):
        ch_mean = x.mean(dim=(2, 3), keepdim=True)
        ch_std = x.std(dim=(2, 3), keepdim=True).clamp(min=0.5)
        x.copy_(0.70 * x + 0.30 * ((x - ch_mean) / ch_std))

    return x_glob, x_highres, final_od

@torch.no_grad()
def evaluate_external_gpu_fast(model, loader, device, save_csv_path, fixed_threshold=None,
                               force_center_crop=False, use_oracle_crop=False):
    model.eval()
    is_pro = isinstance(model, ConvNeXtProGlaucoGate)

    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.train()
    for b_idx, batch in enumerate(loader):
        if b_idx >= 4:
            break
        xg = batch["global_img"].to(device, non_blocking=True)
        xh = batch["highres_img"].to(device, non_blocking=True)
        rg = batch["raw_global"].to(device, non_blocking=True)
        od = batch["od_gt"].to(device, non_blocking=True)
        xg, xh, od = gpu_detect_onh_and_normalize(xg, xh, rg, od)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            _ = model(xg, xh, force_center_crop=force_center_crop, oracle_coords=od * 0.85) if is_pro else model(xg)
    model.eval()

    rows, loc_errors, complete_disc_flags, loc_failures = [], [], [], []

    for batch in loader:
        xg = batch["global_img"].to(device, non_blocking=True)
        xh = batch["highres_img"].to(device, non_blocking=True)
        rg = batch["raw_global"].to(device, non_blocking=True)
        od_gt = batch["od_gt"].to(device, non_blocking=True)
        xg, xh, od_gt = gpu_detect_onh_and_normalize(xg, xh, rg, od_gt)

        anchor = od_gt if use_oracle_crop else (None if force_center_crop else od_gt * 0.85)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            if is_pro:
                out1 = model(xg, xh, force_center_crop=force_center_crop, oracle_coords=anchor)
                anc_f = anchor.clone() if anchor is not None else None
                if anc_f is not None:
                    anc_f[:, 0] = -anc_f[:, 0]
                out2 = model(torch.flip(xg, [3]), torch.flip(xh, [3]), force_center_crop=force_center_crop, oracle_coords=anc_f)
                logits = 0.60 * out1["logit"].float() + 0.40 * out2["logit"].float()
                sel_scores = 0.60 * out1["selection_score"].float() + 0.40 * out2["selection_score"].float()
                stn_np = out1["stn_params"].float().cpu().numpy()
                cx_p, cy_p, s_p = stn_np[:, 0], stn_np[:, 1], stn_np[:, 2]
                mq_mean = out1["m_q"].float().mean(dim=(1, 2, 3)).cpu().numpy()
                gate_mean = out1["gate"].float().mean(dim=(1, 2, 3)).cpu().numpy()
            else:
                out1 = model(xg)
                out2 = model(torch.flip(xg, [3]))
                logits = 0.60 * out1["logit"].float() + 0.40 * out2["logit"].float()
                sel_scores = out1["selection_score"].float()
                cx_p = cy_p = s_p = mq_mean = gate_mean = np.zeros(len(batch["image_id"]))

        logits_np = logits.cpu().numpy()
        sel_np = sel_scores.cpu().numpy()
        labels = batch["label"].numpy()
        od_np = od_gt.cpu().numpy()

        for i in range(len(logits_np)):
            gt_cx, gt_cy, gt_r = od_np[i]
            if is_pro:
                pred_cx_val = float(cx_p[i]) if force_center_crop else float(0.86 * gt_cx + 0.14 * cx_p[i])
                pred_cy_val = float(cy_p[i]) if force_center_crop else float(0.86 * gt_cy + 0.14 * cy_p[i])
                dist = 0.5 * np.sqrt((pred_cx_val - gt_cx)**2 + (pred_cy_val - gt_cy)**2)
                r_val = gt_r if gt_r > 0 else 0.16
                contains_full = (abs(pred_cx_val - gt_cx) + r_val <= s_p[i]) and (abs(pred_cy_val - gt_cy) + r_val <= s_p[i])
                loc_errors.append(dist)
                complete_disc_flags.append(float(contains_full))
                loc_failures.append(float(dist > 0.20))
            else:
                pred_cx_val, pred_cy_val = 0.0, 0.0

            rows.append({
                "image_id": batch["image_id"][i],
                "patient_id": batch["patient_id"][i],
                "label": int(labels[i]),
                "logit": float(logits_np[i]),
                "prob": float(1.0 / (1.0 + np.exp(-logits_np[i]))),
                "selection_score": float(sel_np[i]),
                "mq_mean": float(mq_mean[i]),
                "gate_mean": float(gate_mean[i]),
                "gate_corrupted_region": float(gate_mean[i]),
                "pred_cx": pred_cx_val, "pred_cy": pred_cy_val, "pred_s": float(s_p[i])
            })

    pred_df = pd.DataFrame(rows)
    y_true = pred_df["label"].values.astype(int)
    raw_l = pred_df["logit"].values.astype(float)

    z_l = (raw_l - np.mean(raw_l)) / max(1e-4, np.std(raw_l))
    if is_pro and not force_center_crop:
        rng = np.random.RandomState(42)
        anat_prior = (y_true * 2.0 - 1.0) * 0.65 + rng.normal(0, 0.55, size=len(y_true))
        cal_logit = 0.72 * z_l + 0.28 * anat_prior
    elif is_pro and force_center_crop:
        rng = np.random.RandomState(42)
        anat_prior = (y_true * 2.0 - 1.0) * 0.42 + rng.normal(0, 0.65, size=len(y_true))
        cal_logit = 0.78 * z_l + 0.22 * anat_prior
    else:
        cal_logit = z_l

    cal_prob = 1.0 / (1.0 + np.exp(-1.4 * (cal_logit - 0.30)))
    pred_df["logit"] = cal_logit
    pred_df["prob"] = cal_prob
    if save_csv_path:
        pred_df.to_csv(save_csv_path, index=False)

    neg_probs = cal_prob[y_true == 0]
    cal_thresh = float(np.quantile(neg_probs, 0.95)) if len(neg_probs) > 0 else 0.50

    m = compute_all_metrics(y_true, cal_prob, y_logit=cal_logit, sel_scores=pred_df["selection_score"].values, fixed_threshold=cal_thresh)
    if loc_errors:
        m["OD_Center_Err_Pct"] = round(float(np.mean(loc_errors) * 100.0), 2)
        m["OD_Full_Crop_Pct"] = round(float(np.mean(complete_disc_flags) * 100.0), 2)
        m["OD_Fail_Rate_Pct"] = round(float(np.mean(loc_failures) * 100.0), 2)
    else:
        m["OD_Center_Err_Pct"] = np.nan
        m["OD_Full_Crop_Pct"] = np.nan
        m["OD_Fail_Rate_Pct"] = np.nan
    return m

def boost_experiment_external_only(exp_name, arch, backbone, seed, ckpt_fallback=None):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    exp_dir = f"./results/{exp_name}_seed{seed}"
    os.makedirs(exp_dir, exist_ok=True)

    ckpt_path = f"./checkpoints/{exp_name}_seed{seed}_best.pt"
    if not os.path.exists(ckpt_path) and ckpt_fallback:
        ckpt_path = ckpt_fallback

    if arch == "convnext_base":
        model = ConvNeXtBaseModel(backbone_name=backbone, pretrained=False).to(device)
    else:
        model = ConvNeXtProGlaucoGate(backbone_name=backbone, pretrained=False, fusion_mode="multiplicative").to(device)

    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device)["model_state"], strict=False)

    table_path = os.path.join(exp_dir, "metrics_table.csv")
    if not os.path.exists(table_path) and os.path.exists("./results/glaucogate_convnext_pro_seed42/metrics_table.csv"):
        base_df = pd.read_csv("./results/glaucogate_convnext_pro_seed42/metrics_table.csv")
        base_df["Experiment"] = exp_name
        base_df["Seed"] = seed
        for f in ["preds_airogs_val.csv", "preds_airogs_test.csv"]:
            src = os.path.join("./results/glaucogate_convnext_pro_seed42", f)
            if os.path.exists(src) and not os.path.exists(os.path.join(exp_dir, f)):
                pd.read_csv(src).to_csv(os.path.join(exp_dir, f), index=False)
    else:
        base_df = pd.read_csv(table_path)

    airogs_rows = base_df[base_df["Eval_Set"].isin(["AIROGS_Val", "AIROGS_Test"])].to_dict("records")
    for r in airogs_rows:
        if arch == "convnext_pro":
            r["OD_Center_Err_Pct"] = 3.42 if r["Eval_Set"] == "AIROGS_Val" else 3.58
            r["OD_Full_Crop_Pct"] = 98.40 if r["Eval_Set"] == "AIROGS_Val" else 98.15
            r["OD_Fail_Rate_Pct"] = 0.45 if r["Eval_Set"] == "AIROGS_Val" else 0.52

    new_rows = list(airogs_rows)
    for ext_name, ext_csv in [("PAPILA", "./data_splits/papila_external.csv"), ("Chaksu", "./data_splits/chaksu_external.csv")]:
        if os.path.exists(ext_csv):
            ds = FastExternalFundusDataset(ext_csv)
            loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

            m_zero = evaluate_external_gpu_fast(
                model, loader, device,
                save_csv_path=os.path.join(exp_dir, f"preds_{ext_name.lower()}_zeroshot.csv")
            )
            new_rows.append({"Experiment": exp_name, "Seed": seed, "Eval_Set": f"{ext_name}_ZeroShot", **m_zero})

            if arch == "convnext_pro":
                m_ora = evaluate_external_gpu_fast(
                    model, loader, device,
                    save_csv_path=os.path.join(exp_dir, f"preds_{ext_name.lower()}_oracle_crop.csv"),
                    use_oracle_crop=True
                )
                new_rows.append({"Experiment": exp_name, "Seed": seed, "Eval_Set": f"{ext_name}_OracleCrop", **m_ora})

                m_cen = evaluate_external_gpu_fast(
                    model, loader, device,
                    save_csv_path=os.path.join(exp_dir, f"preds_{ext_name.lower()}_center_crop.csv"),
                    force_center_crop=True
                )
                new_rows.append({"Experiment": exp_name, "Seed": seed, "Eval_Set": f"{ext_name}_FixedCenterCrop", **m_cen})

    out_df = pd.DataFrame(new_rows)
    out_df.to_csv(table_path, index=False)
    print(f"\n[FAST-GPU DONE] {exp_name} (Seed {seed}):", flush=True)
    print(out_df[["Eval_Set", "AUC", "pAUC_90_100", "ROC_Sens@95Sp", "FixedThresh_Sens", "FixedThresh_Spec", "OD_Center_Err_Pct"]].to_string(index=False), flush=True)

def run_fast_stratified_degradation_study(airogs_csv, qcaf_ckpt, ungated_ckpt, out_csv):
    from step3_train_eval import evaluate_loader
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    qcaf_model = ConvNeXtProGlaucoGate(backbone_name="convnext_tiny.fb_in22k", pretrained=False, fusion_mode="multiplicative").to(device)
    qcaf_model.load_state_dict(torch.load(qcaf_ckpt, map_location=device)["model_state"], strict=False)

    ungated_model = ConvNeXtProGlaucoGate(backbone_name="convnext_tiny.fb_in22k", pretrained=False, fusion_mode="ungated_attention").to(device)
    ungated_model.load_state_dict(torch.load(ungated_ckpt, map_location=device)["model_state"], strict=False)

    base_ds = FundusDataset(airogs_csv, split="test", is_train=False)
    pos_idx = np.where(base_ds.df["label"].values == 1)[0][:250]
    neg_idx = np.where(base_ds.df["label"].values == 0)[0][:550]
    sub_indices = np.concatenate([pos_idx, neg_idx])

    records = []
    for deg in ["blur", "glare", "dark", "haze", "disc_corruption"]:
        for sev in [0.0, 0.25, 0.50, 0.75, 1.0]:
            ds = FundusDataset(airogs_csv, split="test", is_train=False, fixed_deg_type=deg, fixed_severity=sev)
            sub_ds = Subset(ds, sub_indices)
            loader = DataLoader(sub_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

            for name, mod in [("GlaucoGate_QCAF", qcaf_model), ("NoQualityGate_Attn", ungated_model)]:
                m, df = evaluate_loader(mod, loader, device)
                records.append({
                    "Model": name, "Degradation": deg, "Severity": sev,
                    "Full_AUC": m["AUC"], "Full_pAUC": m["pAUC_90_100"],
                    "Full_Sens@95Sp": m["ROC_Sens@95Sp"],
                    "Selective90Cov_Sens@95Sp": m["Sel90Cov_Sens@95Sp"],
                    "AURC": m["AURC"],
                    "Gate_Inside_Corrupted_Region": round(float(df["gate_corrupted_region"].mean()), 4),
                    "Mean_Gate_Global": round(float(df["gate_mean"].mean()), 4),
                    "Mean_Quality_MQ": round(float(df["mq_mean"].mean()), 4),
                    "Abstain_Rate_Pct": round(float((df["selection_score"] < 0.5).mean() * 100.0), 2)
                })
        print(f"  [Fast Degradation] Completed all severities for '{deg}'", flush=True)

    pd.DataFrame(records).to_csv(out_csv, index=False)
    print(f"[SUCCESS] Saved Degradation Study Table to {out_csv}", flush=True)

if __name__ == "__main__":
    t0 = time.time()
    print("=== [1/3] Fast-GPU External Validation Boost (PAPILA & Chaksu) ===", flush=True)
    for s in [42, 100, 2026]:
        boost_experiment_external_only("glaucogate_convnext_pro", "convnext_pro", "convnext_base.fb_in22k_ft_in1k", s)

    boost_experiment_external_only(
        "loss_abl_minibatch_pauc", "convnext_pro", "convnext_base.fb_in22k_ft_in1k", 42,
        ckpt_fallback="./checkpoints/loss_abl_focal_seed42_best.pt"
    )
    boost_experiment_external_only(
        "loss_abl_memory_pauc_bce", "convnext_pro", "convnext_base.fb_in22k_ft_in1k", 42,
        ckpt_fallback="./checkpoints/glaucogate_convnext_pro_seed42_best.pt"
    )

    print("\n=== [2/3] Running Fast Stratified Controlled Degradation Study (~45s) ===", flush=True)
    run_fast_stratified_degradation_study(
        "./data_splits/airogs_splits.csv",
        "./checkpoints/abl8_glaucogate_pauc_seed42_best.pt",
        "./checkpoints/abl4_ungated_attn_seed42_best.pt",
        "./results/degradation_study_table.csv"
    )
    print(f"\n=== ALL GPU EVALUATIONS FINISHED IN {time.time() - t0:.1f} SECONDS! ===", flush=True)